# 🧠 AI vs Real Image Classifier — Training Notebook

| | |
|---|---|
| **Architecture** | EfficientNetB3, two-phase fine-tuning |
| **Dataset** | ai-generated-images-vs-real-images (Kaggle) |
| **Input** | Dataset zip from Google Drive (read-only) |

> ⚠️ **Before running:** `Runtime → Change runtime type → T4 GPU`

## Step 1 — Environment Setup

In [ ]:
import subprocess
import tensorflow as tf

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('GPU detected' if gpu_info.returncode == 0 else 'No GPU — switch runtime')
print('GPUs:', tf.config.list_physical_devices('GPU'))

subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'Pillow'], check=True)

import os, json, random, shutil, io, glob, warnings, zipfile
import numpy as np
from PIL import Image, ImageFile, ImageFilter
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score

# ── Config ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS_P1  = 12
EPOCHS_P2  = 20
LR_P1      = 3e-4
LR_P2      = 2e-5
SEED       = 42
DATASET_ROOT         = '/content/dataset'
MODEL_PATH           = '/content/model_v1.keras'
CLASS_JSON_PATH      = '/content/class_names.json'
CONFIDENCE_THRESHOLD = 0.60
TRAIN_FRAC = 0.75
VAL_FRAC   = 0.15

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)
print(f'TensorFlow {tf.__version__} | {"GPU" if gpus else "CPU"}')

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive as gdrive
gdrive.mount('/content/drive', force_remount=False)

DRIVE_ZIP_HINTS = [
    '/content/drive/MyDrive/ai-generated-images-vs-real-images.zip',
    '/content/drive/MyDrive/dataset.zip',
]

ZIP_PATH = next((h for h in DRIVE_ZIP_HINTS if os.path.exists(h)), None)

if ZIP_PATH is None:
    for root_dir, _, files in os.walk('/content/drive/MyDrive'):
        for fname in files:
            if fname.endswith('.zip') and any(kw in fname.lower() for kw in ('ai','real','fake','image','dataset')):
                ZIP_PATH = os.path.join(root_dir, fname); break
        if ZIP_PATH: break

if ZIP_PATH is None:
    raise FileNotFoundError('Could not find dataset zip on Drive. Update DRIVE_ZIP_HINTS.')

print(f'Zip found: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1024**3:.2f} GB)')

## Step 3 — Scan, Balance & Extract Dataset

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp')
ImageFile.LOAD_TRUNCATED_IMAGES = True

def is_image(p): return p.lower().endswith(IMG_EXTS)

def detect_class(fpath):
    parts = fpath.replace('\\', '/').split('/')
    for p in parts[:-1]:
        if p.lower() == 'real': return 'real'
        if p.lower() in ('fake','ai','artificial','generated','synthetic'): return 'ai'
    base = parts[-1].lower()
    if 'real' in base: return 'real'
    if any(k in base for k in ('fake','ai_gen')): return 'ai'
    return None

print('Scanning zip...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    all_entries = zf.namelist()

image_entries = [e for e in all_entries if is_image(e) and not e.startswith('__MACOSX')]
print(f'Image entries: {len(image_entries):,}')

categorized = {'ai': [], 'real': []}
for fp in image_entries:
    cls = detect_class(fp)
    if cls: categorized[cls].append(fp)

for cls, files in categorized.items():
    print(f'  {cls}: {len(files):,}')

# Balance & split
min_count = min(len(v) for v in categorized.values())
for cls in categorized:
    random.shuffle(categorized[cls])
    categorized[cls] = categorized[cls][:min_count]

n_train = int(min_count * TRAIN_FRAC)
n_val   = int(min_count * VAL_FRAC)
splits = {
    'train': {c: categorized[c][:n_train]              for c in categorized},
    'val':   {c: categorized[c][n_train:n_train+n_val] for c in categorized},
    'test':  {c: categorized[c][n_train+n_val:]        for c in categorized},
}
for sp, cls in splits.items():
    print(f'{sp}: {sum(len(v) for v in cls.values()):,}')

# Extract
for split in splits:
    for cls in ['ai','real']:
        os.makedirs(f'{DATASET_ROOT}/{split}/{cls}', exist_ok=True)

already_done = all(
    len(os.listdir(f'{DATASET_ROOT}/{sp}/{cls}')) > 10
    for sp in splits for cls in ['ai','real']
    if os.path.exists(f'{DATASET_ROOT}/{sp}/{cls}')
)

if not already_done:
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for sp, classes in splits.items():
            for cls, files in classes.items():
                dst_dir = f'{DATASET_ROOT}/{sp}/{cls}'
                for i, zpath in enumerate(tqdm(files, desc=f'{sp}/{cls}')):
                    dst = os.path.join(dst_dir, f'{i:06d}_{os.path.basename(zpath)}')
                    if os.path.exists(dst): continue
                    try:
                        with zf.open(zpath) as src: data = src.read()
                        Image.open(io.BytesIO(data)).verify()
                        with open(dst, 'wb') as f: f.write(data)
                    except: pass
    print('Done.')
else:
    print('Already extracted.')

## Step 4 — Augmentation & Generators

In [ ]:
def apply_jpeg_compression(img):
    if random.random() < 0.6:
        q = random.randint(50, 92)
        pil = Image.fromarray(img.astype(np.uint8))
        buf = io.BytesIO(); pil.save(buf, format='JPEG', quality=q); buf.seek(0)
        return np.array(Image.open(buf)).astype(np.float32)
    return img

def apply_gaussian_noise(img):
    if random.random() < 0.4:
        sigma = random.uniform(0.5, 8.0)
        return np.clip(img + np.random.normal(0, sigma, img.shape), 0, 255).astype(np.float32)
    return img

def real_world_aug(img):
    img = apply_jpeg_compression(img)
    img = apply_gaussian_noise(img)
    return img

# Phase 1 — lightweight
datagen_p1 = ImageDataGenerator(rescale=1./255, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen_p1 = datagen_p1.flow_from_directory(
    f'{DATASET_ROOT}/train', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', classes=['ai','real'], shuffle=True, seed=SEED)

val_gen = val_datagen.flow_from_directory(
    f'{DATASET_ROOT}/val', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', classes=['ai','real'], shuffle=False)

# Phase 2 — heavy
datagen_p2 = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=15, zoom_range=0.15,
    width_shift_range=0.1, height_shift_range=0.1, brightness_range=[0.65,1.35],
    fill_mode='reflect', preprocessing_function=real_world_aug)

train_gen_p2 = datagen_p2.flow_from_directory(
    f'{DATASET_ROOT}/train', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', classes=['ai','real'], shuffle=True, seed=SEED)

print(f'Train: {train_gen_p1.samples:,} | Val: {val_gen.samples:,}')
print(f'Class map: {train_gen_p1.class_indices}')

## Step 5 — Build EfficientNetB3

In [ ]:
base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs, name='EffNetB3_AIvsReal')
model.compile(
    optimizer=Adam(learning_rate=LR_P1, clipnorm=1.0),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)
print('Model ready')

## Step 6 — Phase 1: Train Head (Base Frozen)

In [ ]:
callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, min_delta=0.001, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('/content/ckpt_p1.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]
history_p1 = model.fit(train_gen_p1, epochs=EPOCHS_P1, validation_data=val_gen, callbacks=callbacks_p1)
print(f'Phase 1 done | best val_acc={max(history_p1.history["val_accuracy"]):.4f}')

## Step 7 — Phase 2: Fine-Tune Top 50 Layers

In [ ]:
model.load_weights('/content/ckpt_p1.keras')
base.trainable = True
for layer in base.layers[:-50]: layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=LR_P2, clipnorm=1.0),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.03),
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)
callbacks_p2 = [
    EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    ModelCheckpoint('/content/ckpt_best.keras', monitor='val_auc', save_best_only=True, verbose=1),
]
history_p2 = model.fit(train_gen_p2, epochs=EPOCHS_P2, validation_data=val_gen, callbacks=callbacks_p2)
print(f'Phase 2 done | best val_auc={max(history_p2.history["val_auc"]):.4f}')

## Step 8 — Evaluate

In [ ]:
model.load_weights('/content/ckpt_best.keras')

val_gen.reset()
y_prob = model.predict(val_gen, verbose=1).flatten()
y_pred = (y_prob > 0.5).astype(int)
y_true = val_gen.classes

final_acc = accuracy_score(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true, y_prob)
final_auc = auc(fpr, tpr)
print(f'Val Accuracy: {final_acc:.4f} | AUC: {final_auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['AI','REAL'], yticklabels=['AI','REAL'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={final_auc:.4f}')
axes[1].plot([0,1],[0,1],'k--'); axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=['AI','REAL']))

## Step 9 — TTA Inference

In [ ]:
def preprocess_image(path, apply_aug=False):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    if apply_aug and random.random() > 0.5: arr = arr[:, ::-1, :]
    return arr / 255.0

def predict_image(image_path, tta_steps=8, threshold=0.6):
    probs = [float(model.predict(np.expand_dims(
        preprocess_image(image_path, apply_aug=(i>0)), 0), verbose=0)[0][0])
        for i in range(tta_steps)]
    real_prob = float(np.mean(probs))
    ai_prob   = 1.0 - real_prob
    if real_prob >= threshold:   label, conf = 'REAL', real_prob
    elif ai_prob >= threshold:   label, conf = 'AI', ai_prob
    else:                        label, conf = 'UNCERTAIN', max(real_prob, ai_prob)
    emoji = {'AI':'🤖','REAL':'📷','UNCERTAIN':'❓'}[label]
    print(f'{emoji} {label} — confidence: {conf:.1%} | AI: {ai_prob:.1%} | Real: {real_prob:.1%}')
    return {'label':label,'confidence':round(conf,4),'ai_prob':round(ai_prob,4),'real_prob':round(real_prob,4)}

# Quick visual test
import numpy as np
val_gen.reset()
filepaths = np.array(val_gen.filepaths)
labels    = val_gen.classes
sample_idx = np.random.choice(len(filepaths), 4, replace=False)
label_map = {0:'AI', 1:'REAL'}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, idx in zip(axes, sample_idx):
    path = filepaths[idx]
    true_label = label_map[labels[idx]]
    res = predict_image(path)
    ax.imshow(Image.open(path).resize((224,224)))
    color = 'green' if res['label'] == true_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {res['label']} ({res['confidence']:.0%})", color=color)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Step 10 — Save Model & Backup to Drive

In [ ]:
model.save(MODEL_PATH)
print(f'Model saved: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1024**2:.1f} MB)')

class_config = {
    'class_names': ['ai','real'], 'class_indices': {'ai':0,'real':1},
    'label_map': {'0':'AI','1':'REAL'}, 'confidence_threshold': CONFIDENCE_THRESHOLD,
    'model_architecture': 'EfficientNetB3', 'input_size': IMG_SIZE, 'tta_steps_default': 8,
    'final_metrics': {'val_accuracy': round(final_acc,4), 'val_auc': round(final_auc,4)},
}
with open(CLASS_JSON_PATH, 'w') as f: json.dump(class_config, f, indent=2)

# Backup to Drive
BACKUP = '/content/drive/MyDrive/AI_vs_Real_Classifier'
os.makedirs(BACKUP, exist_ok=True)
for src in [MODEL_PATH, CLASS_JSON_PATH]:
    if os.path.exists(src):
        shutil.copy2(src, f'{BACKUP}/{os.path.basename(src)}')
        print(f'Backed up: {os.path.basename(src)}')

# Download locally
from google.colab import files as colab_files
for path in [MODEL_PATH, CLASS_JSON_PATH]:
    colab_files.download(path)
    print(f'Downloaded: {os.path.basename(path)}')